# GDP Ocean Drifters — FlowNP Evaluation & Prediction VisualisationThis notebook loads a trained **FlowNP** (Flow Matching Neural Process)checkpoint and performs a comprehensive evaluation on the **GDP**test set (1,000 held-out ocean drifter snapshots, each recording eastward velocity, northward velocity, and sea-surface temperature at 64–220 irregularly spaced buoy positions on the unit sphere).### How FlowNP predictions workUnlike Gaussian-head models (CNP, ConvCNP) that output $(\mu, \sigma)$analytically, FlowNP generates predictions by **Euler-integrating a learnedvelocity field** from Gaussian noise.  Multiple independent trajectories arerun and aggregated:- **Predicted mean** = average of $S$ sample trajectories.- **Predicted std** = standard deviation across the $S$ trajectories.This gives empirical uncertainty estimates that can capture **non-Gaussian,multi-modal** predictive distributions — a key advantage of FlowNP overGaussian-output NPs.### Test protocolEach test sample is split deterministically (seeded by index) into 30%context / 70% target.  All metrics are computed on the target set.### Unique aspects of the GDP dataset- **Variable sample sizes**: 64–220 drifter observations per snapshot,  requiring padded batching with boolean masks.- **Spherical geometry**: positions are $(x,y,z)$ on the unit sphere.  All  spatial plots use a lat/lon equirectangular projection for readability.- **Three feature channels**: eastward velocity (ve), northward velocity (vn),  and sea-surface temperature (SST) — velocity channels are typically harder  to predict due to turbulent ocean dynamics.**Contents**| § | Section | What it measures ||---|---------|------------------|| 1 | Load checkpoint & build model | — || 2 | Aggregate metrics (RMSE, MAE) | Overall and per-channel prediction accuracy || 2b | Distributional metrics (SWD, MMD) | Whether the predicted distribution matches the data distribution || 3 | Calibration analysis | Whether predicted confidence intervals are honest || 4 | Spatial prediction maps | Geographic patterns in predictions, errors, and uncertainty || 5 | Predicted-mean vs ground-truth scatter | Systematic biases and per-channel accuracy || 6 | Error vs predicted uncertainty | Whether the model "knows what it doesn't know" || 7 | Residual distributions | Distributional shape of normalised errors || 8 | Confidence-interval band plots | Per-sample ±1σ/±2σ prediction envelopes |

In [ ]:
import sys, os, mathsys.path.insert(0, os.path.abspath(".."))import numpy as npimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltimport torchfrom torch.utils.data import DataLoaderfrom scipy.stats import norm as scipy_normfrom scipy.spatial.distance import cdistfrom data import SpatialDataset, spatial_collatefrom model import FlowNPplt.rcParams.update({"figure.dpi": 130, "font.size": 11})def xyz_to_latlon(pos):    lat = np.degrees(np.arcsin(np.clip(pos[:, 2], -1, 1)))    lon = np.degrees(np.arctan2(pos[:, 1], pos[:, 0]))    return lat, lon

---## 1. Load checkpoint & build modelEvery checkpoint stores a `config` dictionary alongside the model weights.Point `CKPT_PATH` at any run's output and the model will be reconstructedautomatically — no need to remember which hyperparameters were used.The test dataset uses `deterministic=True` and a fixed 30% context fractionfor exact reproducibility across evaluation runs.

In [ ]:
CKPT_PATH = "../checkpoints/gdp/best_model.pt"   # ← change to evaluate a different runDATA_PATH = "/home/zross/proj/spl/datasets/gdp_irregular_xyz.npz"EVAL_CONTEXT_FRAC = 0.3DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"NUM_SAMPLES = 20   # trajectories for FlowNP samplingckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)cfg = ckpt["config"]print(f"Checkpoint : {CKPT_PATH}")print(f"Epoch      : {ckpt['epoch']}")print(f"Test loss  : {ckpt.get('test_loss', 'N/A')}")print(f"Config     : {cfg['dataset_name']}, d_model={cfg['d_model']}, "      f"layers={cfg['num_layers']}, heads={cfg['nhead']}")

In [ ]:
model = FlowNP(    dim_x=cfg["x_dim"], dim_y=cfg["y_dim"],    dim_posenc=cfg["dim_posenc"], d_model=cfg["d_model"],    emb_depth=cfg["emb_depth"], dim_feedforward=cfg["dim_feedforward"],    nhead=cfg["nhead"], dropout=cfg.get("dropout", 0.0),    num_layers=cfg["num_layers"], timesteps=cfg.get("timesteps", 100),).to(DEVICE)model.load_state_dict(ckpt["model"])model.eval()print(f"Model loaded — {model.num_params/1e6:.2f}M parameters")

In [ ]:
data_npz = np.load(DATA_PATH, allow_pickle=True)PARAM_NAMES = list(data_npz["parameters"])PARAM_LABELS = ["Eastward vel. (ve)", "Northward vel. (vn)", "SST"]FEATURE_MEAN = data_npz["feature_mean"]FEATURE_STD  = data_npz["feature_std"]test_ds = SpatialDataset(    data_npz["x_test"], data_npz["pos_test"],    context_frac_range=(EVAL_CONTEXT_FRAC, EVAL_CONTEXT_FRAC),    deterministic=True,)test_loader = DataLoader(    test_ds, batch_size=16, shuffle=False,    collate_fn=spatial_collate, num_workers=0,)print(f"Test samples: {len(test_ds)}, features: {PARAM_NAMES}")sizes = [data_npz["x_test"][i].shape[0] for i in range(len(test_ds))]print(f"Points per sample — min: {min(sizes)}, max: {max(sizes)}, "      f"mean: {np.mean(sizes):.0f}")

---## 2. Aggregate metrics over the full test setWe run FlowNP inference (Euler integration with `NUM_SAMPLES` trajectories)on every test sample and pool all valid target-point predictions.  Twodeterministic metrics are computed **per channel** and **overall**:| Metric | Formula | Why it matters ||--------|---------|----------------|| **RMSE** (Root Mean Squared Error) | $\sqrt{\frac{1}{N}\sum_i(y_i-\hat{\mu}_i)^2}$ | Point-prediction accuracy.  Sensitive to large errors.  Lower is better. || **MAE** (Mean Absolute Error) | $\frac{1}{N}\sum_i |y_i-\hat{\mu}_i|$ | Robust accuracy — less sensitive to outliers than RMSE.  A large gap between RMSE and MAE signals occasional large errors.  Lower is better. |> **Note on NLL:** FlowNP uses flow matching, not a Gaussian likelihood.  We> report an *approximate* Gaussian NLL using the empirical mean and std from> the samples.  This is useful for comparison with Gaussian-output baselines> but is not the native training objective.> **Units:** all features are pre-normalised in the `.npz`, so RMSE/MAE> values are in standard-deviation units of the original feature.

In [ ]:
all_true, all_mean, all_std, all_pos = [], [], [], []all_xc, all_yc = [], []with torch.no_grad():    for batch in test_loader:        batch = {k: v.to(DEVICE) for k, v in batch.items()}        mean, std = model.predict(            batch["x_context"], batch["y_context"],            batch["x_target"],            num_samples=NUM_SAMPLES,            ctx_mask=batch["context_mask"],            tar_mask=batch["target_mask"],        )        mask = batch["target_mask"]        B = mean.shape[0]        for i in range(B):            m = mask[i]            all_true.append(batch["y_target"][i][m].cpu().numpy())            all_mean.append(mean[i][m].cpu().numpy())            all_std.append(std[i][m].cpu().numpy())            all_pos.append(batch["x_target"][i][m].cpu().numpy())Y     = np.concatenate(all_true, axis=0)MU    = np.concatenate(all_mean, axis=0)SIGMA = np.concatenate(all_std, axis=0)POS   = np.concatenate(all_pos, axis=0)print(f"Collected {Y.shape[0]:,} target points across {len(all_true)} samples")

In [ ]:
from torch.distributions import Normaln_channels = Y.shape[1]residuals = Y - MUnll_per_ch, rmse_per_ch, mae_per_ch = [], [], []for c in range(n_channels):    dist = Normal(torch.tensor(MU[:, c]), torch.tensor(SIGMA[:, c]).clamp(min=1e-6))    nll_per_ch.append(-dist.log_prob(torch.tensor(Y[:, c])).mean().item())    rmse_per_ch.append(np.sqrt(np.mean(residuals[:, c] ** 2)))    mae_per_ch.append(np.mean(np.abs(residuals[:, c])))dist_all = Normal(torch.tensor(MU), torch.tensor(SIGMA).clamp(min=1e-6))nll_overall  = -dist_all.log_prob(torch.tensor(Y)).mean().item()rmse_overall = np.sqrt(np.mean(residuals ** 2))mae_overall  = np.mean(np.abs(residuals))print(f"{'Channel':20s}  {'NLL*':>8s}  {'RMSE':>8s}  {'MAE':>8s}")print("─" * 50)for c in range(n_channels):    print(f"{PARAM_LABELS[c]:20s}  {nll_per_ch[c]:8.4f}  "          f"{rmse_per_ch[c]:8.4f}  {mae_per_ch[c]:8.4f}")print("─" * 50)print(f"{'Overall':20s}  {nll_overall:8.4f}  "      f"{rmse_overall:8.4f}  {mae_overall:8.4f}")print("\n* NLL is approximate (Gaussian fit to FlowNP's empirical samples)")

---## 2b. Distributional metrics — Sliced Wasserstein Distance & MMDPoint-wise metrics (RMSE, MAE) evaluate each target independently.**Distributional metrics** ask: does the model reproduce the correct*distribution* of values — the right histogram shape, spread, andinter-channel correlations?### Sliced Wasserstein Distance (SWD)The Wasserstein distance (Earth Mover's Distance) measures the minimum costof transforming one distribution into another.  The **sliced** variantprojects both distributions onto many random 1D directions and averages theclosed-form 1D Wasserstein distances:$$\text{SWD}(P, Q) = \mathbb{E}_{\theta \sim \mathcal{S}^{d-1}}\left[ W_1(P_\theta, Q_\theta) \right]$$**Why it matters:** A model could achieve reasonable RMSE by predicting safe,conservative means, yet fail to reproduce the true variability.  SWD detectsthis because it penalises any mismatch in the distribution — not just the mean.**How to interpret:**- SWD = 0 means the two distributions are identical.- Smaller values are better.- Compare "Mean vs Truth" against "Sampled vs Truth": if sampled is lower,  the model's uncertainty ($\sigma$) correctly adds variability that matches  the data.### Maximum Mean Discrepancy (MMD)MMD measures distributional distance in a Reproducing Kernel Hilbert Spaceusing an RBF kernel $k(x,y) = \exp(-\|x-y\|^2 / 2\ell^2)$:$$\text{MMD}^2(P,Q) = \mathbb{E}_{x,x'\sim P}[k(x,x')] + \mathbb{E}_{y,y'\sim Q}[k(y,y')] - 2\,\mathbb{E}_{x\sim P,\,y\sim Q}[k(x,y)]$$The bandwidth $\ell$ is set via the **median heuristic** (median of allpairwise distances).  MMD = 0 iff $P = Q$.**How to interpret:**- MMD is sensitive to *all moments* of the distribution (mean, variance,  skewness, kurtosis, and beyond).- Lower is better.- The median heuristic makes it scale-free — values are comparable across  channels and datasets.### What we compareFor each test snapshot:- **Ground truth** $\{y_j\}$ — real observations.- **Predicted means** $\{\mu_j\}$ — FlowNP's averaged point estimates.- **Predictive samples** $\{\hat{y}_j \sim \mathcal{N}(\mu_j, \sigma_j^2)\}$  — draws from the empirical predictive distribution.

In [ ]:
def sliced_wasserstein(X, Y, n_projections=500):    """Sliced Wasserstein Distance between two point clouds."""    D = X.shape[1]    directions = np.random.randn(n_projections, D)    directions /= np.linalg.norm(directions, axis=1, keepdims=True)    swd = 0.0    for theta in directions:        proj_x = np.sort(X @ theta)        proj_y = np.sort(Y @ theta)        n = max(len(proj_x), len(proj_y))        px = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(proj_x)), proj_x)        py = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(proj_y)), proj_y)        swd += np.mean(np.abs(px - py))    return swd / n_projectionsdef mmd_rbf(X, Y, bandwidth=None):    """MMD with RBF kernel (median heuristic bandwidth)."""    XX = cdist(X, X, 'sqeuclidean')    YY = cdist(Y, Y, 'sqeuclidean')    XY = cdist(X, Y, 'sqeuclidean')    if bandwidth is None:        all_dists = np.concatenate([XX.ravel(), YY.ravel(), XY.ravel()])        bandwidth = np.sqrt(np.median(all_dists[all_dists > 0]) / 2.0)    gamma = 1.0 / (2.0 * bandwidth ** 2)    mmd_sq = np.exp(-gamma * XX).mean() + np.exp(-gamma * YY).mean() - 2 * np.exp(-gamma * XY).mean()    return np.sqrt(max(mmd_sq, 0.0))

In [ ]:
np.random.seed(0)SAMPLES = MU + SIGMA * np.random.randn(*MU.shape)swd_mean_list, swd_sample_list = [], []mmd_mean_list, mmd_sample_list = [], []for y_true, mu, sigma in zip(all_true, all_mean, all_std):    y_sample = mu + sigma * np.random.randn(*mu.shape)    swd_mean_list.append(sliced_wasserstein(y_true, mu))    swd_sample_list.append(sliced_wasserstein(y_true, y_sample))    mmd_mean_list.append(mmd_rbf(y_true, mu))    mmd_sample_list.append(mmd_rbf(y_true, y_sample))print(f"{'Metric':30s}  {'Mean±Std (across samples)':>25s}")print("─" * 60)print(f"{'SWD (mean vs truth)':30s}  "      f"{np.mean(swd_mean_list):.4f} ± {np.std(swd_mean_list):.4f}")print(f"{'SWD (sampled vs truth)':30s}  "      f"{np.mean(swd_sample_list):.4f} ± {np.std(swd_sample_list):.4f}")print(f"{'MMD (mean vs truth)':30s}  "      f"{np.mean(mmd_mean_list):.4f} ± {np.std(mmd_mean_list):.4f}")print(f"{'MMD (sampled vs truth)':30s}  "      f"{np.mean(mmd_sample_list):.4f} ± {np.std(mmd_sample_list):.4f}")

In [ ]:
print(f"\n{'Channel':20s}  {'SWD(μ)':>8s}  {'SWD(s)':>8s}  {'MMD(μ)':>8s}  {'MMD(s)':>8s}")print("─" * 62)for c in range(n_channels):    swd_m = sliced_wasserstein(Y[:, c:c+1], MU[:, c:c+1])    swd_s = sliced_wasserstein(Y[:, c:c+1], SAMPLES[:, c:c+1])    mmd_m = mmd_rbf(Y[:, c:c+1], MU[:, c:c+1])    mmd_s = mmd_rbf(Y[:, c:c+1], SAMPLES[:, c:c+1])    print(f"{PARAM_LABELS[c]:20s}  {swd_m:8.4f}  {swd_s:8.4f}  {mmd_m:8.4f}  {mmd_s:8.4f}")swd_m_all = sliced_wasserstein(Y, MU)swd_s_all = sliced_wasserstein(Y, SAMPLES)mmd_m_all = mmd_rbf(Y, MU)mmd_s_all = mmd_rbf(Y, SAMPLES)print("─" * 62)print(f"{'Overall':20s}  {swd_m_all:8.4f}  {swd_s_all:8.4f}  {mmd_m_all:8.4f}  {mmd_s_all:8.4f}")print("\n(μ) = predicted mean vs truth    (s) = predictive sample vs truth")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))axes[0].hist(swd_mean_list, bins=40, alpha=0.6, color="steelblue",             edgecolor="white", label="Mean vs Truth")axes[0].hist(swd_sample_list, bins=40, alpha=0.6, color="coral",             edgecolor="white", label="Sampled vs Truth")axes[0].set_xlabel("SWD"); axes[0].set_ylabel("# test samples")axes[0].set_title("Per-sample Sliced Wasserstein Distance")axes[0].legend(); axes[0].grid(True, alpha=0.2)axes[1].hist(mmd_mean_list, bins=40, alpha=0.6, color="steelblue",             edgecolor="white", label="Mean vs Truth")axes[1].hist(mmd_sample_list, bins=40, alpha=0.6, color="coral",             edgecolor="white", label="Sampled vs Truth")axes[1].set_xlabel("MMD"); axes[1].set_ylabel("# test samples")axes[1].set_title("Per-sample Maximum Mean Discrepancy")axes[1].legend(); axes[1].grid(True, alpha=0.2)fig.suptitle("Distribution of per-sample SWD and MMD", fontsize=13, y=1.02)plt.tight_layout(); plt.show()

---## 3. Calibration analysisA model is **calibrated** if its predicted confidence intervals contain thetrue value at the stated frequency.  For example, the predicted 90% interval($\mu \pm 1.645\sigma$) should contain the truth for approximately 90% ofpredictions.### How to read the calibration curve- **x-axis**: expected coverage level (0% to 100%).- **y-axis**: observed (empirical) coverage — the fraction of targets that  actually fall inside the interval.- **Perfect calibration**: the curve lies on the diagonal $y = x$.- **Below the diagonal**: the model is **overconfident** — its intervals are  too narrow.- **Above the diagonal**: the model is **conservative** — intervals are wider  than necessary.For FlowNP, the empirical $\sigma$ comes from the variance of multiplesample trajectories.  If the flow generates well-spread samples, calibrationshould be reasonable even though it was not explicitly optimised for (unlikeNLL-trained models).### Expected Calibration Error (ECE)We also report the **ECE** — the average absolute gap between observed andexpected coverage across all levels.  ECE = 0 means perfect calibration.

In [ ]:
levels = np.linspace(0.01, 0.99, 50)z_scores = scipy_norm.ppf(0.5 + levels / 2)fig, ax = plt.subplots(figsize=(7, 7))ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect calibration")for c in range(n_channels):    observed = []    for z in z_scores:        inside = np.abs(residuals[:, c]) <= z * SIGMA[:, c]        observed.append(inside.mean())    ax.plot(levels, observed, lw=2, label=PARAM_LABELS[c])# Overallobserved_all = []for z in z_scores:    inside = np.all(np.abs(residuals) <= z * SIGMA, axis=1)    observed_all.append(inside.mean())ax.plot(levels, observed_all, "k-", lw=2.5, label="Overall (joint)")ece = np.mean(np.abs(np.array(observed_all) - levels))ax.set_xlabel("Expected coverage")ax.set_ylabel("Observed coverage")ax.set_title(f"Calibration curve — ECE = {ece:.4f}")ax.legend(fontsize=9)ax.set_aspect("equal")ax.grid(True, alpha=0.2)plt.tight_layout(); plt.show()print(f"Expected Calibration Error (ECE): {ece:.4f}")

---## 4. Spatial prediction maps (lat/lon projection)For the GDP dataset, positions are $(x,y,z)$ on the unit sphere.  We convertto latitude/longitude for visualisation.### What to look for- **Prediction map**: do spatial patterns match known oceanographic features?- **Error map**: are errors randomly scattered or concentrated in specific  ocean regions (e.g. western boundary currents)?- **Uncertainty map**: does the model assign higher uncertainty to regions far  from context drifters or in turbulent areas?

In [ ]:
def plot_spatial_gdp(model, dataset, sample_idx, channel=2, device=DEVICE):    sample = dataset[sample_idx]    with torch.no_grad():        mean, std = model.predict(            sample["x_context"].unsqueeze(0).to(device),            sample["y_context"].unsqueeze(0).to(device),            sample["x_target"].unsqueeze(0).to(device),            num_samples=NUM_SAMPLES,        )    m, s = mean[0].cpu().numpy(), std[0].cpu().numpy()    yt = sample["y_target"].numpy()    xt = sample["x_target"].numpy()    xc = sample["x_context"].numpy()    lat_t, lon_t = xyz_to_latlon(xt)    lat_c, lon_c = xyz_to_latlon(xc)    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))    titles = ["Ground truth", "Predicted mean", "|Error|", "Predicted σ"]    values = [yt[:, channel], m[:, channel],              np.abs(yt[:, channel] - m[:, channel]), s[:, channel]]    cmaps = ["RdYlBu_r", "RdYlBu_r", "Reds", "Purples"]    for ax, title, val, cmap in zip(axes, titles, values, cmaps):        sc = ax.scatter(lon_t, lat_t, c=val, s=25, cmap=cmap,                        edgecolors="k", linewidths=0.3, zorder=2)        ax.scatter(lon_c, lat_c, c="gray", s=10, marker="x",                   alpha=0.5, zorder=1, label="Context")        ax.set_title(title); plt.colorbar(sc, ax=ax, fraction=0.046)        ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")    fig.suptitle(f"Sample #{sample_idx} — {PARAM_LABELS[channel]} "                 f"(ctx={len(xc)}, tgt={len(xt)})", fontsize=13, y=1.03)    plt.tight_layout(); plt.show()

In [ ]:
for idx in [0, 100, 500]:    plot_spatial_gdp(model, test_ds, idx, channel=2)

In [ ]:
for ch in range(n_channels):    plot_spatial_gdp(model, test_ds, 100, channel=ch)

---## 5. Predicted-mean vs ground-truth scatterA fundamental diagnostic: we plot every predicted mean against thecorresponding ground truth.### How to read the scatter plot- **Tight cluster around $y = x$**: accurate predictions.- **Spread**: quantified by RMSE (in the title).- **Systematic bias**: the cloud curving away from $y = x$.- **Heteroscedastic patterns** (wider spread at extreme values): the model  finds rare conditions harder to predict.

In [ ]:
fig, axes = plt.subplots(1, n_channels, figsize=(5.5 * n_channels, 5))if n_channels == 1: axes = [axes]for c, ax in enumerate(axes):    ax.scatter(Y[:, c], MU[:, c], s=1, alpha=0.12, rasterized=True)    lims = [min(Y[:, c].min(), MU[:, c].min()),            max(Y[:, c].max(), MU[:, c].max())]    ax.plot(lims, lims, "r--", lw=1, label="y = x")    ax.set_xlabel("Ground truth"); ax.set_ylabel("Predicted mean")    ax.set_title(f"{PARAM_LABELS[c]}\nRMSE={rmse_per_ch[c]:.4f}")    ax.set_aspect("equal"); ax.legend(fontsize=9); ax.grid(True, alpha=0.2)fig.suptitle("Predicted mean vs ground truth — all test targets",             fontsize=13, y=1.02)plt.tight_layout(); plt.show()

---## 6. Error vs predicted uncertaintyThis diagnostic answers: **does the model know what it doesn't know?**### How to read the plot- Each dot is one target point; x = predicted $\sigma$, y = $|y - \mu|$.- The **red line** is the bin-averaged absolute error.- The **dashed black line** is the theoretically perfect relationship for a  Gaussian: $\mathbb{E}[|y-\mu|] = \sigma\sqrt{2/\pi} \approx 0.80\,\sigma$.### What to look for- Red line **above** dashed → the model is **overconfident** (predicts  lower uncertainty than warranted).- Red line **below** dashed → the model is **conservative**.- A monotonically increasing red line means the model's uncertainty is at  least **rank-ordered** correctly (high $\sigma$ → high error), even if  the magnitudes are off.

In [ ]:
fig, axes = plt.subplots(1, n_channels, figsize=(5.5 * n_channels, 5))if n_channels == 1: axes = [axes]for c, ax in enumerate(axes):    abs_err = np.abs(residuals[:, c])    ax.scatter(SIGMA[:, c], abs_err, s=1, alpha=0.08, rasterized=True)    n_bins = 30    bins = np.linspace(SIGMA[:, c].min(), np.percentile(SIGMA[:, c], 99), n_bins)    dig = np.digitize(SIGMA[:, c], bins)    bin_means = [abs_err[dig == b].mean() if (dig == b).sum() > 0 else np.nan                 for b in range(1, n_bins)]    bin_centers = 0.5 * (bins[:-1] + bins[1:])    ax.plot(bin_centers, bin_means, "r-", lw=2, label="Binned mean |error|")    xx = np.linspace(0, bins[-1], 100)    ax.plot(xx, xx * np.sqrt(2 / np.pi), "k--", lw=1, alpha=0.6,            label=r"$\sigma\sqrt{2/\pi}$ (perfect)")    ax.set_xlabel("Predicted σ"); ax.set_ylabel("|Error|")    ax.set_title(PARAM_LABELS[c])    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)fig.suptitle("Error vs predicted uncertainty", fontsize=13, y=1.02)plt.tight_layout(); plt.show()

---## 7. Residual distributionsIf FlowNP's empirical mean and std form a reasonable Gaussian approximation,the **standardised residuals** $z_i = (y_i - \mu_i) / \sigma_i$ shouldbe approximately $\mathcal{N}(0,1)$.### What to look for| Statistic | Ideal | Meaning if off ||-----------|-------|----------------|| **mean** | 0 | Non-zero → systematic bias in the predicted mean || **std** | 1 | >1 → overconfident (underestimates uncertainty); <1 → conservative || **shape** | Bell curve | Heavy tails → true errors have more extremes than a Gaussian predicts |Note: FlowNP does **not** assume Gaussian outputs.  This analysis checks howwell a Gaussian approximation (mean/std from samples) captures the actualerror distribution.  Deviations from Gaussianity here are expected and do notindicate a problem — they confirm that FlowNP is modelling something richerthan a Gaussian.

In [ ]:
z_residuals = residuals / np.clip(SIGMA, 1e-6, None)x_ref = np.linspace(-4, 4, 200)pdf_ref = scipy_norm.pdf(x_ref)fig, axes = plt.subplots(1, n_channels, figsize=(5.5 * n_channels, 4.5))if n_channels == 1: axes = [axes]for c, ax in enumerate(axes):    ax.hist(z_residuals[:, c], bins=80, density=True, alpha=0.6,            color="steelblue", edgecolor="white", linewidth=0.3,            label="Standardised residuals")    ax.plot(x_ref, pdf_ref, "r-", lw=2, label="$\\mathcal{N}(0,1)$")    ax.set_xlabel("$(y - \\mu) / \\sigma$")    ax.set_title(f"{PARAM_LABELS[c]}\n"                 f"mean={z_residuals[:, c].mean():.3f}, "                 f"std={z_residuals[:, c].std():.3f}")    ax.legend(fontsize=9); ax.set_xlim(-5, 5); ax.grid(True, alpha=0.2)fig.suptitle("Standardised residuals — comparison with N(0,1)",             fontsize=13, y=1.03)plt.tight_layout(); plt.show()

---## 8. Confidence-interval band plotsFor ocean drifter data, we sort targets by **latitude** and plot ±1σ/±2σbands.### What to look for- **SST**: should follow a clear latitudinal gradient (warm tropics, cold  poles) — bands should widen at latitudes with fewer drifters.- **Velocities (ve, vn)**: ocean currents are more turbulent and chaotic  than SST — expect wider bands and more outliers.- **Red dots outside ±2σ**: a few (~4.6%) are expected.  Many more indicate  underestimated uncertainty.

In [ ]:
def plot_confidence_bands_gdp(model, dataset, sample_idx, channel=2, device=DEVICE):    sample = dataset[sample_idx]    with torch.no_grad():        m, s = model.predict(            sample["x_context"].unsqueeze(0).to(device),            sample["y_context"].unsqueeze(0).to(device),            sample["x_target"].unsqueeze(0).to(device),            num_samples=NUM_SAMPLES,        )    m, s = m[0].cpu().numpy(), s[0].cpu().numpy()    yt = sample["y_target"].numpy()    xt = sample["x_target"].numpy()    xc = sample["x_context"].numpy()    lat_t, _ = xyz_to_latlon(xt)    lat_c, _ = xyz_to_latlon(xc)    order = np.argsort(lat_t)    lat_sorted = lat_t[order]    fig, ax = plt.subplots(figsize=(13, 5))    ax.fill_between(lat_sorted, m[order, channel] - 2*s[order, channel],                    m[order, channel] + 2*s[order, channel],                    alpha=0.15, color="C0", label="μ ± 2σ (95.4%)")    ax.fill_between(lat_sorted, m[order, channel] - s[order, channel],                    m[order, channel] + s[order, channel],                    alpha=0.3, color="C0", label="μ ± 1σ (68.3%)")    ax.plot(lat_sorted, m[order, channel], "C0-", lw=2, label="Predicted mean")    ax.scatter(lat_sorted, yt[order, channel], c="red", s=15, zorder=5,               label="Target truth")    ax.scatter(lat_c, sample["y_context"][:, channel].numpy(), c="gray",               s=10, marker="x", zorder=4, label="Context", alpha=0.6)    ax.set_xlabel("Latitude (°)")    ax.set_ylabel(f"{PARAM_LABELS[channel]} (normalised)")    ax.set_title(f"Sample #{sample_idx} — {PARAM_LABELS[channel]} "                 f"(ctx={len(xc)}, tgt={len(xt)})")    ax.legend(fontsize=9, loc="upper right"); ax.grid(True, alpha=0.2)    plt.tight_layout(); plt.show()

In [ ]:
for idx in [0, 100, 500]:    plot_confidence_bands_gdp(model, test_ds, idx, channel=2)

In [ ]:
for ch in range(n_channels):    plot_confidence_bands_gdp(model, test_ds, 100, channel=ch)

---## Summary### Metrics at a glance| Metric | Scope | What it tells you ||--------|-------|-------------------|| **RMSE** | Overall + per-channel | Point-prediction accuracy (comparable to deterministic baselines) || **MAE** | Overall + per-channel | Robust accuracy (less outlier-sensitive than RMSE) || **NLL*** | Overall + per-channel | Approximate joint quality of mean and uncertainty (Gaussian fit) || **SWD** | Overall + per-channel + per-sample | Whether the predicted distribution matches the data distribution (projection-based) || **MMD** | Overall + per-channel + per-sample | Whether the predicted distribution matches the data distribution (kernel-based) || **Calibration (ECE)** | Overall + per-channel | Honesty of predicted confidence intervals at all coverage levels || **Error vs σ** | Per-channel | Whether high-uncertainty regions truly have higher errors || **Standardised residuals** | Per-channel | Gaussian approximation quality (mean≈0, std≈1, bell shape) |*\*NLL is approximate for FlowNP — computed from the empirical Gaussian fit to FlowNP's samples.*### Visualisations at a glance| Plot | Purpose ||------|---------|| **Spatial maps** (§4) | Geographic patterns in predictions, errors, and uncertainty || **Scatter plots** (§5) | Systematic biases and heteroscedastic patterns per channel || **CI band plots** (§8) | How prediction envelopes widen in data-sparse regions || **SWD/MMD histograms** (§2b) | Per-sample distributional fidelity |### ReproducingAll splits are deterministic.  Use the same checkpoint with`EVAL_CONTEXT_FRAC = 0.3` and `deterministic=True`.